In [1]:
from oqd_compiler_infrastructure import (
    FixedPoint,
    InplacePre,
    Post,
    PrettyPrint,
    cfg_to_dot,
)
from rich import print as pprint

from oqd_core.analog import AnalogCFGBuilder, parse_analog, serialize_analog
from oqd_core.analysis.analog.bounds_checker import AnalogBoundsChecker
from oqd_core.analysis.analog.dim_checker import DimensionChecker
from oqd_core.analysis.analog.fold import ConstantFolding, ConstantFoldingRule
from oqd_core.analysis.analog.reaching_def import AvailableVariableAnalysis
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.analysis.dominator import PostDominatorAnalysis
from oqd_core.compiler.analog import AnalogCFGtoAST

printer = Post(PrettyPrint())

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()


circuit = parse_analog(source)
cfg = AnalogCFGBuilder()(circuit)
# pprint(cfg)

In [2]:
avail_checker = AvailableVariableAnalysis()
avail_res = avail_checker.analyze(cfg)

In [3]:
type_checker = AnalogTypeChecker()
type_res = type_checker.analyze(cfg)

In [4]:
bounds_checker = AnalogBoundsChecker()
bounds_res = bounds_checker.analyze(cfg)

In [5]:
dim_checker = DimensionChecker()
dim_res = dim_checker.analyze(cfg)

In [6]:
fold_checker = ConstantFolding()
fold_res = fold_checker.analyze(cfg)
# pprint(fold_res)

fold_pass = FixedPoint(InplacePre(ConstantFoldingRule(fold_res=fold_res)))

fold_pass(cfg)

CFG(blocks={0: CFGBlock(register_id=0, stmts=[], preds=set(), succs={1}, exit_nodes=set(), edge_labels={}, tags={'__label__': 'entry'}), 1: CFGBlock(register_id=1, stmts=[Declaration(class_='Declaration', name='r', value=QuantumRegister(class_='QuantumRegister', size=Constant(class_='Constant', value=2), dim=Constant(class_='Constant', value=2)))], preds={0}, succs={2}, exit_nodes=set(), edge_labels={}, tags={}), 2: CFGBlock(register_id=2, stmts=[Declaration(class_='Declaration', name='s', value=ModeRegister(class_='ModeRegister', size=Constant(class_='Constant', value=3)))], preds={1}, succs={3}, exit_nodes=set(), edge_labels={}, tags={}), 3: CFGBlock(register_id=3, stmts=[Declaration(class_='Declaration', name='s2', value=ModeRegister(class_='ModeRegister', size=Constant(class_='Constant', value=3)))], preds={2}, succs={4}, exit_nodes=set(), edge_labels={}, tags={}), 4: CFGBlock(register_id=4, stmts=[Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', value=A

In [7]:
dot = cfg_to_dot(cfg, serialize=serialize_analog, max_lines=5)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)

digraph {
	0 [label=" Block #0
------------------------
--------- tags ---------
__label__: entry"]
	0 -> 1
	1 [label=" Block #1
------------------------
r = qreg(2, 2)"]
	1 -> 2
	2 [label=" Block #2
------------------------
s = qmode(3)"]
	2 -> 3
	3 [label=" Block #3
------------------------
s2 = qmode(3)"]
	3 -> 4
	4 [label=" Block #4
------------------------
q0 = r[0]"]
	4 -> 5
	5 [label=" Block #5
------------------------
q1 = r[1]"]
	5 -> 6
	6 [label=" Block #6
------------------------
targets = [q0, q1]"]
	6 -> 7
	7 [label=" Block #7
------------------------
pi = 3.14159"]
	7 -> 8
	8 [label=" Block #8
------------------------
tau = 6.28318"]
	8 -> 9
	9 [label=" Block #9
------------------------
omega = #omega"]
	9 -> 10
	10 [label=" Block #10
------------------------
phase = #t * omega + #phi"]
	10 -> 11
	11 [label=" Block #11
------------------------
neg = -1"]
	11 -> 12
	12 [label=" Block #12
------------------------
cubed = 8"]
	12 -> 13
	13 [label=" Block #13
----------------

'cfg.png'

In [8]:
pdom_result = PostDominatorAnalysis().analyze(cfg)

circuit = AnalogCFGtoAST(pdom_result=pdom_result)(cfg)
int
with open("reserialized_test.analog", "w") as f:
    f.write(serialize_analog(circuit))